In [ ]:
from binn import BINN
import pandas as pd
import numpy as np
import torch
import random
import pickle
from binn.DICE import DICE
test_input_data = pd.read_csv("/home/zxl/hdd/cellfate/data/TCGA_primary_tpm_100.csv")
test_input_sign = pd.read_csv("/home/zxl/hdd/cellfate/data/subtype_primary_tpm_100.csv")

with open('/home/zxl/hdd/cellfate/data/Gene_and_network1.pkl', 'rb') as file:
    data = pickle.load(file)
gene_list = data["gene_list"]
gene_set = set(gene_list.tolist())  
binn = BINN(
    activation = "tanh",
    activation_final = "sigmoid",
    connectivity_matrices_list = data,
    dropout=0.2,
    validate=False,
    device="cuda:0",
    learning_rate=0.001,
) 
trainer = DICE(binn)
return_dict= trainer.fit(test_input_data,
                        test_input_sign,
                        nr_iterations=1,
                        temperature_init=1.0,
                        connectivity_matrices_list = data,
                        batch_size=32,
                        n_folds=3,
                        val_size = 0.2,
                        test_size = 0.2,
                        max_epochs=100,
                        num_workers=0,
                        gene_list=gene_list)




/home/zxl/.conda/envs/DICE/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/zxl/hdd/cellfate/binn/binn.py:198: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.loss = nn.CrossEntropyLoss(weight=torch.tensor(self.weight, device=device))



BINN is on the device: cuda:0
BINN(
  (batchnorm_np): BatchNorm1d(7057, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (B_cell_layers): Sequential(
    (Layer_0): Linear(in_features=7057, out_features=1044, bias=True)
    (BatchNorm_0): BatchNorm1d(1044, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (Dropout_0): Dropout(p=0.2, inplace=False)
    (Tanh 0): Tanh()
    (Layer_1): Linear(in_features=1044, out_features=522, bias=True)
    (BatchNorm_1): BatchNorm1d(522, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (Dropout_1): Dropout(p=0.2, inplace=False)
    (Tanh 1): Tanh()
    (Layer_2): Linear(in_features=522, out_features=121, bias=True)
    (BatchNorm_2): BatchNorm1d(121, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (Dropout_2): Dropout(p=0.2, inplace=False)
    (Tanh 2): Tanh()
    (Layer_3): Linear(in_features=121, out_features=16, bias=True)
    (BatchNorm_3): BatchNorm1d(16, eps=1e-05, momentu

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores



BINN is on the device: cuda:0


/home/zxl/.conda/envs/DICE/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
You are using a CUDA device ('NVIDIA GeForce RTX 4090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/home/zxl/.conda/envs/DICE/lib/python3.10/site-packa

Optimized Temperature: 1.2784


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
     Validate metric           DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
         val_acc            0.9582608938217163
        val_loss            0.23692725598812103
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/home/zxl/.conda/envs/DICE/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=77` in the `DataLoader` to improve performance.


────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
         test_F1            0.9679017663002014
        test_acc            0.9710144996643066
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


In [ ]:
import pandas as pd
data=return_dict
metrics = {
        'iteration': data['iteration'],
        'train_acc': [x[0] for x in data['train_acc']],
        'train_loss': [x[0] for x in data['train_loss']],
        'val_acc': [x[0] for x in data['val_acc']],
        'val_loss': [x[0] for x in data['val_loss']],
        'test_acc': [x[0] for x in data['test_acc']],
        'test_f1': [x[0] for x in data['test_f1']],
        'epochs': [x[0] for x in data['epochs']]
    }  
df = pd.DataFrame(metrics)
df.to_csv("/model/metrics.csv", index=False)

In [ ]:
import torch
import pandas as pd
import pickle
import numpy as np
from sklearn.metrics import classification_report, f1_score, recall_score, precision_score
input_data = pd.read_csv("/data/MET500_621_exp.csv")
label_df = pd.read_csv("/data/MET500_621_group.csv")  
with open('/data/Gene_and_network1.pkl', 'rb') as file:
    data = pickle.load(file)
gene_list = data["gene_list"]
def _fit_data_matrix_to_network_input(data_matrix, features, feature_column="Gene"):

    if len(features) > len(data_matrix.index):
        features_df = pd.DataFrame(features, columns=[feature_column])
        data_matrix = data_matrix.merge(features_df, how="right", on=feature_column)
    if len(features) > 0:
        data_matrix.set_index(feature_column, inplace=True)
        data_matrix = data_matrix.loc[features]
    return data_matrix
def _data_pre(data_matrix, design_matrix, groups): 
    y_list, dfs, sample_names = [], [], []
    for group in groups:
        group_samples = design_matrix[design_matrix["group"] == group]["sample"].values
        df_group = data_matrix[group_samples].T 
        dfs.append(df_group)
        y_list += [group - 1] * len(group_samples) 
        sample_names.extend(group_samples)
    
    X = pd.concat(dfs).fillna(0).to_numpy()
    y = np.array(y_list)
    return X, y, sample_names

fitted_input = _fit_data_matrix_to_network_input(
    input_data.reset_index(), 
    features=gene_list, 
    feature_column="Gene"
)
X_new, y_new, samples = _data_pre(
    fitted_input, 
    design_matrix=label_df, 
    groups=np.arange(1, 25)
)

device="cudo:0"
X_new_tensor = torch.tensor(X_new, dtype=torch.float32).to(device)
y_new_tensor = torch.tensor(y_new, dtype=torch.long).to(device)
model = torch.load('/model/binn_model0_tpm_0.001_32.pth', map_location="cuda:1", weights_only=False)

with torch.no_grad():
    new_preds = model(X_new_tensor)
    new_probs, new_labels = torch.max(new_preds, dim=1)

result_df = pd.DataFrame({
    "sample": samples,
    "correct": (new_labels.cpu().numpy() == y_new).astype(int),
    "true": y_new,
    "pred": new_labels.cpu().numpy()
})
result_df.to_csv('/home/zxl/hdd/cellfate/result/prediction_results_MET500.csv', index=False)



In [ ]:
from sklearn.metrics import classification_report, f1_score, recall_score, precision_score
import pandas as pd 
import numpy as np
import csv

prediction_results_TCGA1165 = pd.read_csv('/result/prediction_results_TCGAMET.csv')
prediction_results_bcgsc = pd.read_csv('/result/prediction_results_bcgsc.csv')
prediction_results_ICGC = pd.read_csv('/result/prediction_results_ICGC.csv')
prediction_results_MET500 = pd.read_csv('/result/prediction_results_MET500.csv')
def eval(pred):
    Y_pre_label_change = pred['pred'] 
    Y_true_label = pred['ture'] 
    ture = pred['correct']
    print('ACC_score',sum((ture==1))/len(ture))
    print('f1_score', f1_score(Y_pre_label_change, Y_true_label,average='weighted'))    
    print('precision_score', precision_score(Y_pre_label_change, Y_true_label, average='weighted', zero_division=0))
    print('recall_score', recall_score(Y_pre_label_change, Y_true_label, average='weighted', zero_division=0))

    print(sum((ture==1)))
    print(len(ture))



eval(prediction_results_TCGA1165)
print('*'*100)
eval(prediction_results_bcgsc)
print('*'*100)
eval(prediction_results_ICGC)
print('*'*100)
eval(prediction_results_MET500)




In [ ]:
import pickle
import torch
from binn import BINN, Network
import pandas as pd
import numpy as np
import torch
import random
from binn.explain import SHAPExplainer
test_input_data = pd.read_csv("/data/TCGA_primary_tpm_100.csv")
test_input_sign = pd.read_csv("/data/subtype_primary_tpm_100.csv")
import gc
model_path = f'/model/binn_model_tpm_0.001_32.pth'
model = torch.load(model_path, map_location="cuda:1",weights_only=False)
        
      
shap = SHAPExplainer(
            input_data=test_input_data, 
            design_matrix=test_input_sign, 
            model=model,
            device="cuda:1"
        )

shap.explain(
            output_dir="/model", 
            iteration=0
   
        )
    

In [ ]:
import pickle
import torch
from binn import BINN, Network
import pandas as pd
import numpy as np
import torch
import random
from binn.explain import SHAPExplainer
import gc
test_input_data = pd.read_csv("/data/TCGA_primary_tpm_100.csv")
test_input_sign = pd.read_csv("/data/subtype_primary_tpm_100.csv")

model_path = f'/model/binn_model_tpm_0.001_32.pth'
model = torch.load(model_path, map_location="cuda:0",weights_only=False)
        
      
shap = SHAPExplainer(
            input_data=test_input_data, 
            design_matrix=test_input_sign, 
            model=model,
            device="cuda:0"
        )

shap.explain_cell(
            output_dir="/model", 
            iteration=0

        )